# Preparing Training Data

First of all, we need to convert the training data from .json to .spacy (so that we can train our model)

In [ ]:
import re
import spacy
from spacy.tokenizer import Tokenizer

prefix_re = re.compile(r'[(\[\'"]|<i>')
suffix_re = re.compile(r'[.,;:?!)\]\'"]|</i>[.,;:?!)\]\'"]*')
infix_re = re.compile(r'[-/+]')

def custom_tokenizer(nlp):
    return Tokenizer(
        nlp.vocab,
        prefix_search=prefix_re.search,
        suffix_search=suffix_re.search,
        infix_finditer=infix_re.finditer
    )

In [ ]:
def prepare_data(json_files):
    db = DocBin()
    for json_file in json_files:
        counter = 0
        print('Parsing {}'.format(json_file))
        f = open(json_file)
        data = json.load(f)
    
        for article in data:
            title = data[article]['metadata']['title']
            abstract = data[article]['metadata']['abstract']
            entities = data[article]['entities']
    
            title_doc = nlp(title)
            abstract_doc = nlp(abstract)
    
            title_ents = []
            abstract_ents = []
            for entity in entities:
                start = int(entity['start_idx'])
                end = int(entity['end_idx']) + 1
                label = entity['label']

                expected_text = title[start:end] if entity["location"] == "title" else abstract[start:end]
                
                if entity['location'] == 'title':
                    span = title_doc.char_span(start, end, label=label, alignment_mode='strict')
                    if span:
                        title_ents.append(span)
                    else:
                        print('-'*60)
                        print(title)
                        print('Tokens -> ', [(i, token) for i, token in enumerate(title_doc)])
                        print(f'Expected Entity: {expected_text} (Label: {label}, Start: {start}, End: {end})')
                elif entity['location'] == 'abstract':
                    span = abstract_doc.char_span(start, end, label=label, alignment_mode='strict') # with alignment_mode='strict' only exact matches
                    if span and span.text != 'D': # to remove D span, which is in conflict with Vitamin D
                        abstract_ents.append(span)
                    else:
                        print('-'*60)
                        print(title)
                        print('Tokens -> ', [(i, token) for i, token in enumerate(abstract_doc)])
                        print(f'Expected Entity: {expected_text} (Label: {label}, Start: {start}, End: {end})')
                else:
                    print('ERROR: {}'.format(entity['location']))
    
            title_doc.ents = title_ents
            abstract_doc.ents = abstract_ents
            db.add(title_doc)
            db.add(abstract_doc)
            counter += 1
        print(counter)
    return db

In [ ]:
import json
from spacy.tokens import DocBin

train_json = [
    'gutbrainie2025/Annotations/Train/platinum_quality/json_format/train_platinum.json',
    'gutbrainie2025/Annotations/Train/gold_quality/json_format/train_gold.json',
    'gutbrainie2025/Annotations/Train/silver_quality/json_format/train_silver.json',
    'gutbrainie2025/Annotations/Train/bronze_quality/json_format/train_bronze.json'
]

dev_json = [
    'gutbrainie2025/Annotations/Dev/json_format/dev.json'
]

nlp = spacy.blank('en')
nlp.tokenizer = custom_tokenizer(nlp)

train_db = prepare_data(train_json)
dev_db = prepare_data(dev_json)
train_db.to_disk('./train.spacy')
dev_db.to_disk('./dev.spacy')             

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg
!python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy

In [ ]:
nlp = spacy.load('output/model-best/')
doc = nlp("Identification of proteotoxic and proteoprotective bacteria that non-specifically affect proteins associated with neurodegenerative diseases.") # taken from bronze collection

for ent in doc.ents:
    print("{} -> {}".format(ent, ent.label_))